# Silver Pipeline Orchestrator – Price Audit

**Purpose:**

This notebook orchestrates the Silver pipeline for the Price Audit domain, executing in strict sequence the transformation, validation, and monitoring modules. It is designed for Databricks Serverless and Unity Catalog, following enterprise Lakehouse best practices.

- No internal logic of the modules is modified.
- All orchestration is robust, auditable, and modular.
- All code, comments, and docstrings are in English for professional/enterprise review.

**Pipeline Steps:**
1. Silver Transformation
2. Silver Validation
3. Silver Monitoring

**Key Principles:**
- Robust error handling and logging
- Full traceability via `silver_batch_id`
- Configurable, idempotent, and production-ready
- Outputs are ready for external monitoring and alerting (e.g., n8n)


In [ ]:
# SECTION 1: Initial Setup and Configuration
"""
Initializes Spark session, loads configuration, and sets up logging.
All configuration is centralized in the `config` dictionary for flexibility and auditability.
"""

import logging
from pyspark.sql import SparkSession

# Configure logging (enterprise level)
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver_price_audit_orchestrator")

# Centralized configuration object (edit as needed for environment)
config = {
    "write_audit_log": True,  # Enable/disable audit log persistence (set True for production/orchestrator)
    "write_alerts": True,     # Enable/disable validation alerts persistence
    "write_monitoring_log": True,  # Enable/disable monitoring log persistence
    "environment": "prod",   # dev, qa, prod
    # Table overrides (optional)
    # "source_table": "workspace.bronze.price_audit",
    # "target_table": "workspace.silver.fact_price_audit",
    # "alerts_table": "workspace.silver.validation_alerts_price_audit",
    # "monitoring_table": "workspace.silver.pipeline_monitoring_log",
    # "fail_on_severities": ["P0", "P1"]
}

# Initialize Spark session (Databricks Serverless/Connect compatible)
spark = SparkSession.builder.appName("SilverPriceAuditOrchestrator").getOrCreate()

logger.info("Notebook started. Environment: %s", config["environment"])


## Section 2: Orchestration – Silver Transformation

This step executes the Silver transformation module. All output fields are logged and displayed. If the transformation fails, the pipeline is stopped and monitoring is attempted for traceability.

In [ ]:
# SECTION 2: Silver Transformation Orchestration
"""
Executes the Silver transformation step. All output fields are logged and displayed.
If the transformation fails, the pipeline is stopped and monitoring is attempted for traceability.
"""

import sys
import os
import importlib
# Ensure src is in sys.path for module imports
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Dynamic import and reload for robustness in notebooks
module_name = "silver.fact_price_audit.transformation_fact_price_audit"
transformation_module = importlib.import_module(module_name)
importlib.reload(transformation_module)
run_price_audit_transformation = getattr(transformation_module, "run_price_audit_transformation")

# Initialize with safe defaults for robustness
transformation_result = {
    "status": "UNKNOWN",
    "error_message": "Not executed"
}
silver_batch_id = None

try:
    result = run_price_audit_transformation(spark, config)
    if result:
        transformation_result = result
        silver_batch_id = transformation_result.get("silver_batch_id")
        config["silver_batch_id"] = silver_batch_id
    else:
        raise ValueError("Transformation returned None")
    # Log and display all output fields
    logger.info("Transformation output: %s", transformation_result)
    print("\n--- TRANSFORMATION RESULT ---")
    for k, v in transformation_result.items():
        print(f"{k}: {v}")
    if transformation_result["status"] != "SUCCESS":
        logger.error("Transformation failed. Stopping pipeline.")
        raise RuntimeError(f"Transformation failed: {transformation_result['error_message']}")
except Exception as e:
    logger.error(f"[CRITICAL] Transformation step failed: {e}")
    print(f"[CRITICAL] Transformation step failed: {e}")
    # Optionally, attempt monitoring for traceability
    # Optionally, set a flag to skip further steps
    raise


## Section 3: Orchestration – Silver Validation

This step executes the Silver validation module only if the transformation succeeded. All output fields are logged and displayed. If validation fails or should_fail_pipeline is True, the pipeline is marked as failed but monitoring is still executed for traceability.

In [ ]:
# SECTION 3: Silver Validation Orchestration
"""
Executes the Silver validation step only if transformation succeeded.
All output fields are logged and displayed. If validation fails or should_fail_pipeline is True,
the pipeline is marked as failed but monitoring is still executed for traceability.
"""

import importlib
# Dynamic import and reload for robustness in notebooks
validation_module_name = "silver.fact_price_audit.validation_fact_price_audit"
validation_module = importlib.import_module(validation_module_name)
importlib.reload(validation_module)
run_price_audit_validation = getattr(validation_module, "run_price_audit_validation")

validation_result = {
    "status": "UNKNOWN",
    "error_message": "Not executed",
    "total_alerts": 0
}
pipeline_should_fail = False

# If transformation failed, skip validation and set consistent output
if not transformation_result or transformation_result.get("status") != "SUCCESS":
    logger.error("Skipping validation: transformation failed")
    validation_result = {
        "status": "SKIPPED",
        "error_message": "Transformation did not succeed",
        "total_alerts": 0
    }
    pipeline_should_fail = True
else:
    try:
        # Propagate silver_batch_id for traceability
        config["silver_batch_id"] = transformation_result.get("silver_batch_id")
        result = run_price_audit_validation(spark, config)
        if result:
            validation_result = result
        else:
            raise ValueError("Validation returned None")
        logger.info("Validation output: %s", validation_result)
        print("\n--- VALIDATION RESULT ---")
        for k, v in validation_result.items():
            print(f"{k}: {v}")
        if validation_result.get("should_fail_pipeline"):
            logger.warning("Validation indicates pipeline should fail (blocking alert detected).")
            pipeline_should_fail = True
    except Exception as e:
        logger.error(f"[CRITICAL] Validation step failed: {e}")
        print(f"[CRITICAL] Validation step failed: {e}")
        validation_result = validation_result or {"status": "FAILURE", "error_message": str(e), "total_alerts": 0}
        pipeline_should_fail = True


## Section 4: Orchestration – Silver Monitoring

This step consolidates the results of transformation and validation, and executes the monitoring module. All output fields are logged and displayed. The monitoring result is persisted if enabled, and is ready for external consumption (e.g., n8n).

In [ ]:
# SECTION 4: Silver Monitoring Orchestration
"""
Consolidates transformation and validation results, then executes the monitoring module.
All output fields are logged and displayed. Monitoring result is persisted if enabled and is ready for external consumption.
"""

import importlib
# Dynamic import and reload for robustness in notebooks
monitoring_module_name = "silver.fact_price_audit.monitoring_fact_price_audit"
monitoring_module = importlib.import_module(monitoring_module_name)
importlib.reload(monitoring_module)
run_price_audit_monitoring = getattr(monitoring_module, "run_price_audit_monitoring")

# Ensure validation_result is always a valid dict for monitoring
validation_result = validation_result or {
    "status": "UNKNOWN",
    "total_alerts": 0,
    "should_fail_pipeline": False
}

monitoring_result = None

try:
    # Always attempt monitoring for traceability, even if previous steps failed
    monitoring_result = run_price_audit_monitoring(spark, validation_result, config)
    logger.info("Monitoring output: %s", monitoring_result)
    print("\n--- MONITORING RESULT ---")
    for k, v in monitoring_result.items():
        print(f"{k}: {v}")
except Exception as e:
    logger.error(f"[CRITICAL] Monitoring step failed: {e}")
    print(f"[CRITICAL] Monitoring step failed: {e}")
    monitoring_result = monitoring_result or {"pipeline_status": "FAILURE", "error_message": str(e)}


## Section 5: Error Handling, Logging, and Final Pipeline Status

This section summarizes the pipeline status, logs all errors and durations, and ensures that traceability is never lost. The final output is prepared for external monitoring and alerting.

In [ ]:
# SECTION 5: Error Handling, Logging, and Final Pipeline Status
"""
Summarizes pipeline status, logs all errors and durations, and ensures traceability is never lost.
Prepares the final output for external monitoring and alerting.
"""

final_status = "SUCCESS"

# Transformation is mandatory
if not transformation_result or transformation_result.get("status") != "SUCCESS":
    final_status = "FAILURE"
# Validation is conditional, but if executed, must be SUCCESS or SKIPPED
elif validation_result and validation_result.get("status") not in ["SUCCESS", "SKIPPED"]:
    final_status = "FAILURE"
# Explicit pipeline failure flag
elif pipeline_should_fail:
    final_status = "FAILURE"
# Monitoring is observability only, but log if failed
elif monitoring_result and monitoring_result.get("pipeline_status") == "FAILURE":
    final_status = "FAILURE"

logger.info(f"FINAL PIPELINE STATUS: {final_status}")
print(f"\n=== FINAL PIPELINE STATUS: {final_status} ===\n")

# Prepare structured output for external systems (e.g., n8n)
final_output = {
    "transformation": transformation_result,
    "validation": validation_result,
    "monitoring": monitoring_result,
    "final_status": final_status,
    "silver_batch_id": silver_batch_id,
    "environment": config["environment"],
}

import json
try:
    json_output = json.dumps(final_output, indent=2, default=str)
    print("\n--- PIPELINE FINAL OUTPUT (JSON) ---")
    print(json_output)
except Exception as e:
    logger.error(f"Failed to serialize final output to JSON: {e}")
    print("\n--- PIPELINE FINAL OUTPUT (RAW DICT) ---")
    print(final_output)


## Section 6: Persisting Metrics and Auditability

This section ensures that all relevant outputs and metrics are persisted in Delta tables for auditability and traceability, using `silver_batch_id` as the correlation key.

In [ ]:
# SECTION 6: Persisting Metrics and Auditability
"""
Ensures all relevant outputs and metrics are persisted in Delta tables for auditability and traceability,
using silver_batch_id as the correlation key. This is handled by each module if enabled in config.
"""

# No additional code needed here if modules handle persistence via config.
# This section documents the auditability approach for future maintainers and auditors.

print("\nAll metrics and audit logs are persisted by each module if enabled in config.\n")
print(f"silver_batch_id used as correlation key: {silver_batch_id}")


## Section 7: Structured Output and Final Flow Control

This section prepares and displays the final structured output of the pipeline, including all relevant fields for monitoring, alerting, and external consumption. The final pipeline state is determined according to business rules and configuration.

In [ ]:
# SECTION 7: Structured Output and Final Flow Control
"""
Prepares and displays the final structured output of the pipeline, including all relevant fields for monitoring, alerting, and external consumption.
Determines the final pipeline state according to business rules and configuration.
"""

# Final output already prepared in previous section (final_output)
print("\nPipeline execution complete. Review logs and outputs above for full traceability.")
if final_status == "SUCCESS":
    print("\n✅ Pipeline completed successfully. All steps passed.")
else:
    print("\n❌ Pipeline failed. Review errors and monitoring output for details.")
